# Restore checkpoints and reproduce reports

Attach the three original inputs and one complete saved notebook output containing `clip_retrieval/cache` and `clip_retrieval/results`. Select a T4 GPU and turn Internet off. This entry does not train. Existing final-test and timing results are read instead of rerunning those measurements. The validation check is recomputed. Output is copied to a separate working directory.

Generated from the recorded source. Paths and entry-point guards were organized; training equations and measured results were not changed. Python syntax and CPU retrieval checks were run locally; this new notebook has not yet been executed top-to-bottom in a fresh Kaggle GPU kernel.

In [ ]:
# Configure these paths once; all following cells use these aliases.
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
import json
from pathlib import Path

def unique_input(pattern):
    found = sorted(Path("/kaggle/input").rglob(pattern))
    if len(found) != 1:
        raise RuntimeError(f"Expected one {pattern}, found {found}. Check attached inputs.")
    return found[0]

DATA_JSON = Path(os.environ["CLIP_ANNOTATION_PATH"]) if os.environ.get("CLIP_ANNOTATION_PATH") else unique_input("dataset_flickr30k.json")
CLIP_DIR = Path(os.environ["CLIP_MODEL_DIR"]) if os.environ.get("CLIP_MODEL_DIR") else unique_input("pytorch_model.bin").parent
with open(DATA_JSON, encoding="utf-8") as f:
    _configuration_annotation = json.load(f)
IMAGE_ROOT = Path(os.environ["CLIP_IMAGE_DIR"]) if os.environ.get("CLIP_IMAGE_DIR") else unique_input(_configuration_annotation["images"][0]["filename"]).parent
PROJECT_ROOT = Path(os.environ.get("CLIP_OUTPUT_ROOT", "/kaggle/working/clip_retrieval_restored"))
ROOT = PROJECT_ROOT
JSON_PATH, MODEL_DIR, IMG_DIR = DATA_JSON, CLIP_DIR, IMAGE_ROOT
print("Annotation:", DATA_JSON)
print("Images:", IMAGE_ROOT)
print("Offline model:", CLIP_DIR)
print("Output:", PROJECT_ROOT)


## 14 Session file recovery and backup

Original source cell index: 13 (zero-based).

In [ ]:
from pathlib import Path
import hashlib
import shutil
import zipfile
from IPython.display import display, FileLink

print("===== 1. 寻找已挂载的实验输出 =====", flush=True)

required = [
    f"results/{group}_seed{seed}/{filename}"
    for seed in (42, 43, 44)
    for group, filename in (
        ("B_projection", "best_projection.pt"),
        ("C_lora_r8", "best_lora.pt"),
    )
]

candidates = {
    p.parents[2]
    for p in Path("/kaggle/input").rglob("best_projection.pt")
    if p.parent.name == "B_projection_seed42"
    and p.parent.parent.name == "results"
}

valid_sources = [
    folder for folder in sorted(candidates)
    if all(
        (folder / name).is_file()
        and (folder / name).stat().st_size > 0
        for name in required
    )
]

if len(valid_sources) != 1:
    raise RuntimeError(
        f"找到 {len(valid_sources)} 份包含六个权重的输出，"
        f"需要确认来源。候选目录：{sorted(candidates)}"
    )

SOURCE = valid_sources[0]
ROOT = PROJECT_ROOT
print("找到源目录：", SOURCE, flush=True)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


print("\n===== 2. 恢复并校验全部实验文件 =====", flush=True)
source_files = sorted(p for p in SOURCE.rglob("*") if p.is_file())

for source_file in source_files:
    relative = source_file.relative_to(SOURCE)
    target = ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)

    if target.exists():
        # 已有同名文件必须完全一致，避免覆盖不同实验结果
        if sha256_file(source_file) != sha256_file(target):
            raise RuntimeError(f"同名文件内容不同，已停止：{target}")
    else:
        # 不继承只读输入文件的权限，恢复后可继续写入
        shutil.copyfile(source_file, target)
        if sha256_file(source_file) != sha256_file(target):
            raise RuntimeError(f"复制校验失败：{target}")

print(f"已恢复并通过 SHA256 校验：{len(source_files)} 个文件")

for name in required:
    path = ROOT / name
    print(f"{name}: {path.stat().st_size / 1024**2:.2f} MiB")

print("\n===== 3. 生成备份压缩包 =====", flush=True)
backup = (PROJECT_ROOT.parent / (PROJECT_ROOT.name + "_backup.zip"))

with zipfile.ZipFile(
    backup, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1
) as z:
    for source_file in source_files:
        restored = ROOT / source_file.relative_to(SOURCE)
        z.write(restored, arcname=restored.relative_to(ROOT.parent))

with zipfile.ZipFile(backup, "r") as z:
    bad_file = z.testzip()
    if bad_file is not None:
        raise RuntimeError(f"压缩包校验失败：{bad_file}")

print("压缩包完整性检查通过")
print(f"备份大小：{backup.stat().st_size / 1024**2:.2f} MiB")
print("备份路径：", backup)
display(FileLink(str(backup)))

print("\n恢复与备份文件生成完成，请将 ZIP 下载到本地。")

In [ ]:
import json

required_files = [
    "results/final_test/completed.json",
    "results/final_test/test_per_run.csv",
    "results/final_test/test_mean_std.csv",
    "results/final_test/test_metadata.json",
    "results/final_test/test_image_features_fp32.pt",
    "results/complexity/params_and_compute.csv",
    "results/complexity/inference_speed.csv",
    "results/complexity/inference_speed_details.json",
    "results/submission_metadata/saved_experiment_records.json",
]

for label in ("A_baseline", "B_seed42", "C_seed42"):
    required_files.append(
        f"results/final_test/{label}_text_features_fp32.pt"
    )

for name in (
    "fig01_training_and_validation",
    "fig02_test_bidirectional_recall",
    "fig03_trainable_parameters_vs_test_mR",
    "fig04_encoding_latency_and_throughput",
):
    for extension in ("png", "pdf"):
        required_files.append(
            f"results/paper_figures/{name}.{extension}"
        )

for direction in ("T2I", "I2T"):
    for category in (
        "both_top1_success",
        "lora_top1_improvement",
        "lora_top5_failure",
    ):
        required_files.append(
            f"results/retrieval_visualizations/"
            f"{direction}_{category}.png"
        )

missing = [
    name for name in required_files
    if not (ROOT / name).is_file()
    or (ROOT / name).stat().st_size == 0
]

print("检查目录：", ROOT)
print("关键文件数量：", len(required_files))
print("缺失或空文件：", missing)
assert not missing, "关键文件不齐全，先停止后续运行"

with open(
    ROOT / "results/final_test/completed.json",
    encoding="utf-8",
) as f:
    saved_test = json.load(f)

assert len(saved_test["runs"]) == 7

print("\n已恢复的最终测试结果：")
for run in saved_test["runs"]:
    print(f"{run['label']}: mR={run['metrics']['mR']:.4f}%")

print("\n关键成果完整性检查通过，未重新执行测试。")

## 16 Restore model definitions and six checkpoints

Original source cell index: 15 (zero-based).

In [ ]:
print("[恢复] 单元格已开始执行", flush=True)

import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import gc
import json
import math
import hashlib
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor

ROOT = PROJECT_ROOT
MODEL_DIR = CLIP_DIR
DEVICE = torch.device("cuda:0")

assert torch.cuda.is_available(), "请先启用昨天使用的 T4 GPU"
assert (MODEL_DIR / "pytorch_model.bin").is_file(), "离线模型未挂载"


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


# 1. 确保原始CLIP权重与训练时完全一致
print("[恢复] 校验原始CLIP文件，读取约600MB，请稍等", flush=True)

with open(
    ROOT / "results/C_lora_r8_seed42/summary.json",
    encoding="utf-8",
) as f:
    reference_summary = json.load(f)

base_hash = file_sha256(MODEL_DIR / "pytorch_model.bin")
assert base_hash == reference_summary["base_weight_sha256"], (
    "原始CLIP权重与训练时不一致，已停止"
)
print("原始权重SHA256一致", flush=True)

processor = CLIPProcessor.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
    backend="pil",
)


# 2. 恢复相同数学形式的手写LoRA：W(x) + (alpha/r) B(A(x))
class ManualLoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=8):
        super().__init__()
        self.base = base
        self.r = int(r)
        self.scaling = float(alpha) / self.r
        self.enabled = True

        self.base.requires_grad_(False)

        self.lora_A = nn.Parameter(
            base.weight.new_empty(self.r, base.in_features)
        )
        self.lora_B = nn.Parameter(
            base.weight.new_zeros(base.out_features, self.r)
        )
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        output = self.base(x)
        if self.enabled:
            output = output + self.scaling * F.linear(
                F.linear(x, self.lora_A), self.lora_B
            )
        return output


def inject_text_lora(model, r=8, alpha=8):
    count = 0
    for layer in model.text_model.encoder.layers:
        for name in ("q_proj", "v_proj"):
            original = getattr(layer.self_attn, name)
            assert isinstance(original, nn.Linear)
            setattr(
                layer.self_attn,
                name,
                ManualLoRALinear(original, r=r, alpha=alpha),
            )
            count += 1
    assert count == 24, f"LoRA注入数量异常：{count}"
    return count


# 独立的恢复函数，不调用昨天的训练循环
def load_saved_retrieval_model(group, seed=42):
    if group not in ("A", "B", "C"):
        raise ValueError("group必须为A、B或C")

    model = CLIPModel.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
    ).float()
    model.requires_grad_(False)

    if group == "A":
        return model.eval()

    folder_name = (
        f"B_projection_seed{seed}"
        if group == "B"
        else f"C_lora_r8_seed{seed}"
    )
    folder = ROOT / "results" / folder_name

    with open(folder / "summary.json", encoding="utf-8") as f:
        summary = json.load(f)

    assert summary["seed"] == seed
    assert summary["base_weight_sha256"] == base_hash

    checkpoint_name = (
        "best_projection.pt" if group == "B" else "best_lora.pt"
    )
    checkpoint = torch.load(
        folder / checkpoint_name,
        map_location="cpu",
        weights_only=True,
    )
    assert checkpoint["seed"] == seed
    assert checkpoint["epoch"] == summary["best_epoch"]

    if group == "B":
        model.text_projection.requires_grad_(True)
        model.text_projection.load_state_dict(
            checkpoint["projection"], strict=True
        )
        assert torch.equal(
            model.text_projection.weight.detach(),
            checkpoint["projection"]["weight"],
        )
    else:
        r, alpha = int(checkpoint["rank"]), checkpoint["alpha"]
        assert r == summary["rank"]
        assert alpha == summary["alpha"]
        inject_text_lora(model, r=r, alpha=alpha)
        model.text_projection.requires_grad_(True)

        saved = checkpoint["trainable_state"]
        parameters = dict(model.named_parameters())
        expected = {
            name for name, parameter in parameters.items()
            if parameter.requires_grad
        }
        assert set(saved) == expected, (
            f"保存参数与模型不匹配："
            f"缺少={expected - set(saved)}，"
            f"多出={set(saved) - expected}"
        )

        with torch.no_grad():
            for name, value in saved.items():
                parameter = parameters[name]
                assert parameter.shape == value.shape, name
                parameter.copy_(value)
                assert torch.equal(parameter, value), name

    trainable = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    total = sum(p.numel() for p in model.parameters())

    assert trainable == summary["trainable_params"]
    assert total == summary["total_params"]
    assert not any(
        p.requires_grad for p in model.vision_model.parameters()
    )
    assert not any(
        p.requires_grad for p in model.visual_projection.parameters()
    )

    return model.eval()


# 3. 逐份加载六个检查点，并执行一条文本的前向检查
print("\n===== 六份权重加载与前向检查 =====", flush=True)

# 仅用于接口检查，不计算检索指标
probe = processor(
    text=["A person standing outdoors."],
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=77,
)
probe = {key: value.to(DEVICE) for key, value in probe.items()}

restored_c_model = None

for seed in (42, 43, 44):
    for group in ("B", "C"):
        print(f"正在恢复 {group} / seed={seed}", flush=True)
        current = load_saved_retrieval_model(group, seed).to(DEVICE)

        with torch.no_grad():
            pooled = current.text_model(**probe).pooler_output
            feature = F.normalize(
                current.text_projection(pooled), dim=-1
            )

        assert feature.shape == (1, 512)
        assert torch.isfinite(feature).all()

        print(
            f"通过：参数加载一致，特征形状={tuple(feature.shape)}，"
            f"L2范数={feature.norm(dim=-1).item():.6f}",
            flush=True,
        )

        # 保留C组seed42，供下一步复核验证集
        if group == "C" and seed == 42:
            restored_c_model = current

        del current, pooled, feature
        gc.collect()

print("\n===== 恢复完成 =====")
print("processor：已恢复")
print("ManualLoRALinear：已恢复")
print("load_saved_retrieval_model：已恢复，可加载A/B/C")
print("六份权重：均通过加载与文本前向检查")
print("restored_c_model：C组seed42，已加载到cuda:0")
print("未执行训练；未读取测试集。")

## 17 Reproduce the saved validation result

Original source cell index: 16 (zero-based).

In [ ]:
print("[复核] 开始恢复验证集与评估函数", flush=True)

import json
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

JSON_PATH = DATA_JSON

# 1. 核对标注，按缓存中的图片顺序重建caption对应关系
summary_path = ROOT / "results/C_lora_r8_seed42/summary.json"
with open(summary_path, encoding="utf-8") as f:
    saved_summary = json.load(f)

assert file_sha256(JSON_PATH) == saved_summary["annotation_sha256"]

cache = torch.load(
    ROOT / "cache/image_features_fp32.pt",
    map_location="cpu",
    weights_only=True,
)
print("缓存字段：", list(cache.keys()), flush=True)

metadata = cache["metadata"]
val_filenames = list(metadata["val_filenames"])
val_image_features = cache["val"].float().contiguous()

with open(JSON_PATH, encoding="utf-8") as f:
    annotation = json.load(f)

val_by_filename = {
    item["filename"]: item
    for item in annotation["images"]
    if item["split"] == "val"
}

assert len(val_filenames) == len(set(val_filenames)) == 1014
assert set(val_filenames) == set(val_by_filename)
assert val_image_features.shape == (1014, 512)
assert torch.isfinite(val_image_features).all()
assert torch.allclose(
    val_image_features.norm(dim=-1),
    torch.ones(1014),
    atol=1e-5,
)

val_captions = []
val_caption_to_image = []

for image_index, filename in enumerate(val_filenames):
    sentences = val_by_filename[filename]["sentences"]
    assert len(sentences) == 5
    for sentence in sentences:
        caption = sentence["raw"]
        assert isinstance(caption, str) and caption.strip()
        val_captions.append(caption)
        val_caption_to_image.append(image_index)

val_caption_to_image = torch.tensor(
    val_caption_to_image, dtype=torch.long
)

print(f"验证图片：{len(val_filenames)}")
print(f"验证caption：{len(val_captions)}")
print("图片与caption映射：按缓存中的文件名顺序建立", flush=True)


# 2. 恢复全候选库双向评估函数
def metrics_from_ranks(ranks):
    ranks = np.asarray(ranks, dtype=np.int64)
    assert ranks.ndim == 1 and len(ranks) > 0
    assert np.all(ranks >= 1)

    return {
        "R@1": float(100 * np.mean(ranks <= 1)),
        "R@5": float(100 * np.mean(ranks <= 5)),
        "R@10": float(100 * np.mean(ranks <= 10)),
        "MedianRank": float(np.median(ranks)),
    }


@torch.no_grad()
def evaluate_retrieval(image_features, text_features, caption_to_image):
    images = image_features.to(DEVICE, dtype=torch.float32)
    texts = text_features.to(DEVICE, dtype=torch.float32)
    mapping = torch.as_tensor(
        caption_to_image, dtype=torch.long, device=DEVICE
    )

    n_images, n_texts = len(images), len(texts)
    assert mapping.shape == (n_texts,)
    assert int(mapping.min()) >= 0
    assert int(mapping.max()) < n_images
    assert torch.all(torch.bincount(mapping, minlength=n_images) > 0)

    # 一次覆盖完整验证候选库，非batch内评估
    similarity = images @ texts.T
    assert torch.isfinite(similarity).all()

    # I2T：排序后遇到的第一条正确caption，即多正例最小排名
    image_order = torch.argsort(
        similarity, dim=1, descending=True, stable=True
    )
    image_hits = (
        mapping[image_order]
        == torch.arange(n_images, device=DEVICE)[:, None]
    )
    i2t_ranks = image_hits.int().argmax(dim=1) + 1

    del image_order, image_hits

    # T2I：每条caption在全部去重图片中寻找其对应图片
    text_order = torch.argsort(
        similarity.T, dim=1, descending=True, stable=True
    )
    text_hits = text_order == mapping[:, None]
    t2i_ranks = text_hits.int().argmax(dim=1) + 1

    ranks = {
        "I2T": i2t_ranks.cpu().numpy(),
        "T2I": t2i_ranks.cpu().numpy(),
    }
    metrics = {
        direction: metrics_from_ranks(values)
        for direction, values in ranks.items()
    }
    metrics["mR"] = float(np.mean([
        metrics[direction][f"R@{k}"]
        for direction in ("I2T", "T2I")
        for k in (1, 5, 10)
    ]))
    return metrics, ranks


# 3. FP32提取全量验证文本特征
print("\n[复核] 提取完整验证集文本特征", flush=True)

restored_c_model.float().eval()
text_batches = []

with torch.no_grad():
    for start in tqdm(
        range(0, len(val_captions), 64),
        desc="验证文本特征",
    ):
        inputs = processor(
            text=val_captions[start:start + 64],
            padding=True,
            truncation=True,
            max_length=77,
            return_tensors="pt",
        )
        inputs = {
            key: value.to(DEVICE) for key, value in inputs.items()
        }

        pooled = restored_c_model.text_model(**inputs).pooler_output
        features = F.normalize(
            restored_c_model.text_projection(pooled), dim=-1
        )
        text_batches.append(features.cpu())

val_text_features = torch.cat(text_batches, dim=0)
assert val_text_features.shape == (5070, 512)

print("[复核] 在1014张图片、5070条caption上统一排序", flush=True)
recovered_metrics, recovered_ranks = evaluate_retrieval(
    val_image_features,
    val_text_features,
    val_caption_to_image,
)


# 4. 逐项对照昨天保存的结果
print("\n===== C组seed42：恢复前后验证结果 =====")
expected = saved_summary["val_metrics"]
differences = []

for direction in ("I2T", "T2I"):
    for metric in ("R@1", "R@5", "R@10", "MedianRank"):
        old = float(expected[direction][metric])
        new = recovered_metrics[direction][metric]
        differences.append(abs(new - old))
        print(
            f"{direction} {metric:10s} "
            f"原结果={old:.6f}  恢复后={new:.6f}  "
            f"差值={new - old:+.6f}"
        )

mr_difference = abs(recovered_metrics["mR"] - expected["mR"])
differences.append(mr_difference)

print(
    f"mR：原结果={expected['mR']:.6f}%  "
    f"恢复后={recovered_metrics['mR']:.6f}%"
)

max_difference = max(differences)
print(f"全部指标最大绝对差值：{max_difference:.8f}")

assert max_difference < 1e-4, (
    "恢复后的验证指标存在差异，先检查实现与数据顺序，不进入测试集。"
)

print("\n验证集复核通过：恢复后的指标与昨天一致。")
print("未训练；未读取测试集；未覆盖原实验结果。")

## 18 Final test evaluation

Original source cell index: 17 (zero-based).

In [ ]:
TEST_OUT = ROOT / "results" / "final_test"
IMG_DIR = IMAGE_ROOT
if (TEST_OUT / "completed.json").is_file():
    import pandas as pd
    from IPython.display import display
    print("Reading the saved final test result; no test evaluation is rerun.")
    display(pd.read_csv(TEST_OUT / "test_per_run.csv"))
    display(pd.read_csv(TEST_OUT / "test_mean_std.csv"))
else:
    print("[测试] 最终测试集评估已开始", flush=True)

    import gc
    import json
    import hashlib
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn.functional as F
    from pathlib import Path
    from PIL import Image
    from tqdm.auto import tqdm
    from IPython.display import display

    IMG_DIR = IMAGE_ROOT
    TEST_OUT = ROOT / "results" / "final_test"
    TEST_OUT.mkdir(parents=True, exist_ok=True)

    if (TEST_OUT / "completed.json").exists():
        raise RuntimeError(
            "最终测试已完成，结果在 results/final_test。"
            "请直接查看结果，不需要重复运行。"
        )


    def save_json(path, data):
        path = Path(path)
        temporary = path.with_suffix(path.suffix + ".tmp")
        with open(temporary, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        temporary.replace(path)


    # ============================================================
    # 1. 检查划分：只按现有JSON的test字段读取
    # ============================================================
    with open(JSON_PATH, encoding="utf-8") as f:
        raw = json.load(f)

    annotation_hash = file_sha256(JSON_PATH)

    split_ids = {}
    split_filenames = {}

    for split in ("train", "val", "test"):
        records = [x for x in raw["images"] if x["split"] == split]
        split_ids[split] = {x["imgid"] for x in records}
        split_filenames[split] = {x["filename"] for x in records}
        assert len(records) == len(split_ids[split])
        assert len(records) == len(split_filenames[split])

    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        assert split_ids[left].isdisjoint(split_ids[right])
        assert split_filenames[left].isdisjoint(split_filenames[right])

    test_records = sorted(
        [x for x in raw["images"] if x["split"] == "test"],
        key=lambda x: x["filename"],
    )
    assert len(test_records) == 1000

    test_filenames = [x["filename"] for x in test_records]
    test_captions = []
    test_caption_to_image = []

    for image_index, item in enumerate(test_records):
        assert (IMG_DIR / item["filename"]).is_file()
        assert len(item["sentences"]) == 5

        for sentence in item["sentences"]:
            caption = sentence["raw"]
            assert isinstance(caption, str) and caption.strip()
            test_captions.append(caption)
            test_caption_to_image.append(image_index)

    assert len(test_captions) == 5000

    print("\n===== 固定测试候选库 =====")
    print("图片：1000张；caption：5000条；每图5句")
    print("train / val / test 的图片ID与文件名均无交集", flush=True)


    # ============================================================
    # 2. 记录并锁定六份检查点，不根据测试结果选模型
    # ============================================================
    selected_checkpoints = []
    summaries = {}

    for seed in (42, 43, 44):
        for group in ("B", "C"):
            folder_name = (
                f"B_projection_seed{seed}"
                if group == "B"
                else f"C_lora_r8_seed{seed}"
            )
            folder = ROOT / "results" / folder_name

            with open(folder / "summary.json", encoding="utf-8") as f:
                summary = json.load(f)

            assert summary["annotation_sha256"] == annotation_hash
            assert summary["base_weight_sha256"] == base_hash
            assert summary["seed"] == seed

            filename = "best_projection.pt" if group == "B" else "best_lora.pt"
            checkpoint_path = folder / filename
            summaries[(group, seed)] = summary

            selected_checkpoints.append({
                "group": group,
                "seed": seed,
                "best_epoch_from_validation": summary["best_epoch"],
                "checkpoint": str(checkpoint_path.relative_to(ROOT)),
                "checkpoint_sha256": file_sha256(checkpoint_path),
            })

        assert (
            summaries[("B", seed)]["sampling_plan_sha256"]
            == summaries[("C", seed)]["sampling_plan_sha256"]
        ), f"seed={seed}的B/C抽样计划不同"

    test_metadata = {
        "filenames": test_filenames,
        "image_ids": [x["imgid"] for x in test_records],
        "captions": test_captions,
        "caption_to_image": test_caption_to_image,
    }

    manifest = {
        "annotation_sha256": annotation_hash,
        "base_weight_sha256": base_hash,
        "checkpoints": selected_checkpoints,
        "test_metadata_sha256": hashlib.sha256(
            json.dumps(
                test_metadata, ensure_ascii=False, sort_keys=True
            ).encode("utf-8")
        ).hexdigest(),
        "precision": "float32",
        "image_batch_size": 32,
        "text_batch_size": 64,
        "selection": "best validation mR epoch for each predefined seed",
    }

    manifest_path = TEST_OUT / "evaluation_manifest.json"
    if manifest_path.exists():
        with open(manifest_path, encoding="utf-8") as f:
            assert json.load(f) == manifest, "配置与已有测试记录不同，已停止"
    else:
        save_json(manifest_path, manifest)

    save_json(TEST_OUT / "test_metadata.json", test_metadata)
    print("六份检查点与测试配置已记录并锁定", flush=True)


    # ============================================================
    # 3. 特征提取：FP32、eval、无梯度
    # ============================================================
    @torch.no_grad()
    def extract_test_images(model):
        batches = []
        model.float().eval()

        for start in tqdm(
            range(0, len(test_filenames), 32),
            desc="测试图片特征",
        ):
            images = []
            for filename in test_filenames[start:start + 32]:
                with Image.open(IMG_DIR / filename) as image:
                    images.append(image.convert("RGB"))

            pixels = processor(
                images=images, return_tensors="pt"
            )["pixel_values"].to(DEVICE)

            pooled = model.vision_model(
                pixel_values=pixels
            ).pooler_output
            features = F.normalize(
                model.visual_projection(pooled), dim=-1
            )
            batches.append(features.cpu())

        return torch.cat(batches, dim=0)


    @torch.no_grad()
    def extract_test_texts(model, label):
        batches = []
        model.float().eval()

        for start in tqdm(
            range(0, len(test_captions), 64),
            desc=f"{label} 测试文本特征",
        ):
            inputs = processor(
                text=test_captions[start:start + 64],
                padding=True,
                truncation=True,
                max_length=77,
                return_tensors="pt",
            )
            inputs = {
                key: value.to(DEVICE) for key, value in inputs.items()
            }
            pooled = model.text_model(**inputs).pooler_output
            features = F.normalize(
                model.text_projection(pooled), dim=-1
            )
            batches.append(features.cpu())

        return torch.cat(batches, dim=0)


    # 释放之前保留的验证模型显存，后面逐个加载
    restored_c_model.cpu()
    gc.collect()
    torch.cuda.empty_cache()

    runs = [("A", None)] + [
        (group, seed)
        for seed in (42, 43, 44)
        for group in ("B", "C")
    ]

    all_results = []
    rows = []

    # ============================================================
    # 4. 七次完整候选库评估
    # ============================================================
    for group, seed in runs:
        label = "A_baseline" if group == "A" else f"{group}_seed{seed}"
        print(f"\n===== 开始 {label} =====", flush=True)

        current = load_saved_retrieval_model(
            group, seed=42 if seed is None else seed
        ).to(DEVICE).float().eval()

        # 所有组的视觉分支相同且冻结，图片特征只计算一次
        if group == "A":
            test_image_features = extract_test_images(current)
            assert test_image_features.shape == (1000, 512)
            assert torch.isfinite(test_image_features).all()

            torch.save(
                test_image_features,
                TEST_OUT / "test_image_features_fp32.pt",
            )

        text_features = extract_test_texts(current, label)
        assert text_features.shape == (5000, 512)
        assert torch.isfinite(text_features).all()

        metrics, ranks = evaluate_retrieval(
            test_image_features,
            text_features,
            test_caption_to_image,
        )

        result = {
            "label": label,
            "group": group,
            "seed": seed,
            "best_epoch": (
                None if group == "A"
                else summaries[(group, seed)]["best_epoch"]
            ),
            "metrics": metrics,
        }
        all_results.append(result)

        np.savez_compressed(
            TEST_OUT / f"{label}_ranks.npz",
            I2T=ranks["I2T"],
            T2I=ranks["T2I"],
        )

        # 预先固定seed42作为后续检索可视化模型，避免按测试分数挑种子
        if group == "A" or seed == 42:
            torch.save(
                text_features,
                TEST_OUT / f"{label}_text_features_fp32.pt",
            )

        save_json(TEST_OUT / f"{label}_metrics.json", result)

        row = {
            "Model": label,
            "Seed": seed,
            "BestEpoch": result["best_epoch"],
        }
        for direction in ("I2T", "T2I"):
            for key, value in metrics[direction].items():
                row[f"{direction}_{key}"] = value
        row["mR"] = metrics["mR"]
        rows.append(row)

        print("I2T:", metrics["I2T"])
        print("T2I:", metrics["T2I"])
        print(f"mR: {metrics['mR']:.4f}%", flush=True)

        del current, text_features
        gc.collect()
        torch.cuda.empty_cache()


    # ============================================================
    # 5. 保存原始结果及三种子的均值、样本标准差
    # ============================================================
    per_run = pd.DataFrame(rows)
    per_run.to_csv(TEST_OUT / "test_per_run.csv", index=False)

    metric_columns = [
        f"{direction}_{metric}"
        for direction in ("I2T", "T2I")
        for metric in ("R@1", "R@5", "R@10", "MedianRank")
    ] + ["mR"]

    aggregate_rows = []

    for group in ("B", "C"):
        group_rows = [
            row for row in rows if row["Model"].startswith(group + "_")
        ]
        assert len(group_rows) == 3

        for metric in metric_columns:
            values = np.array([row[metric] for row in group_rows])
            aggregate_rows.append({
                "Group": group,
                "Metric": metric,
                "Mean": float(values.mean()),
                "SampleStd": float(values.std(ddof=1)),
                "Seeds": 3,
            })

    aggregate = pd.DataFrame(aggregate_rows)
    aggregate.to_csv(TEST_OUT / "test_mean_std.csv", index=False)

    paired_deltas = np.array([
        next(
            x["metrics"]["mR"] for x in all_results
            if x["group"] == "C" and x["seed"] == seed
        )
        - next(
            x["metrics"]["mR"] for x in all_results
            if x["group"] == "B" and x["seed"] == seed
        )
        for seed in (42, 43, 44)
    ])

    print("\n===== 最终测试集：七次原始结果 =====")
    display(per_run.round(4))

    print("\n===== B/C三种子：均值与样本标准差 =====")
    display(aggregate.round(4))

    print(
        "\nC-B配对mR变化："
        f"{paired_deltas.mean():+.4f} ± "
        f"{paired_deltas.std(ddof=1):.4f} 个百分点"
    )

    save_json(TEST_OUT / "completed.json", {
        "status": "completed",
        "runs": all_results,
        "paired_mR_delta_mean_pp": float(paired_deltas.mean()),
        "paired_mR_delta_sample_std_pp": float(paired_deltas.std(ddof=1)),
    })

    print("\n第12步最终测试集评估完成。")
    print("结果目录：", TEST_OUT)
    print("未训练；未根据测试结果重新选择epoch或种子。")

## 19 Bidirectional retrieval visualizations

Original source cell index: 18 (zero-based).

In [ ]:
print("[可视化] 开始读取已保存的测试特征与排名", flush=True)

import json
import textwrap
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from IPython.display import display

TEST_OUT = ROOT / "results" / "final_test"
VIS_OUT = ROOT / "results" / "retrieval_visualizations"
VIS_OUT.mkdir(parents=True, exist_ok=True)

IMG_DIR = IMAGE_ROOT

assert (TEST_OUT / "completed.json").is_file()

with open(TEST_OUT / "test_metadata.json", encoding="utf-8") as f:
    visual_metadata = json.load(f)

filenames = visual_metadata["filenames"]
captions = visual_metadata["captions"]
caption_to_image = np.asarray(
    visual_metadata["caption_to_image"], dtype=np.int64
)

image_features = torch.load(
    TEST_OUT / "test_image_features_fp32.pt",
    map_location="cpu",
    weights_only=True,
)

labels = {
    "A_baseline": "Original CLIP",
    "C_seed42": "LoRA + projection (seed 42)",
}
scores = {}
saved_ranks = {}

with torch.no_grad():
    for model_name in labels:
        text_features = torch.load(
            TEST_OUT / f"{model_name}_text_features_fp32.pt",
            map_location="cpu",
            weights_only=True,
        )
        assert image_features.shape == (1000, 512)
        assert text_features.shape == (5000, 512)

        scores[model_name] = (
            image_features.to(DEVICE)
            @ text_features.to(DEVICE).T
        ).cpu().numpy()

        with np.load(TEST_OUT / f"{model_name}_ranks.npz") as ranks:
            saved_ranks[model_name] = {
                direction: ranks[direction].copy()
                for direction in ("I2T", "T2I")
            }
        del text_features

GREEN = "#17823b"
RED = "#b23a36"


def read_picture(image_index):
    with Image.open(IMG_DIR / filenames[image_index]) as image:
        return image.convert("RGB")


def ranking_for_query(direction, query_index, model_name):
    matrix = scores[model_name]

    if direction == "T2I":
        order = np.argsort(-matrix[:, query_index], kind="stable")
        correct_image = int(caption_to_image[query_index])
        rank = int(np.flatnonzero(order == correct_image)[0]) + 1
    else:
        order = np.argsort(-matrix[query_index], kind="stable")
        rank = int(
            np.flatnonzero(
                caption_to_image[order] == query_index
            )[0]
        ) + 1

    # 确认用于绘图的排序与正式评估保存的排名一致
    assert rank == int(
        saved_ranks[model_name][direction][query_index]
    ), f"{direction}/{model_name} 的绘图排名与正式结果不一致"

    return order[:5], rank


def draw_t2i(query_index, case_name):
    correct_image = int(caption_to_image[query_index])
    fig = plt.figure(figsize=(18, 7))
    grid = fig.add_gridspec(2, 6, wspace=0.12, hspace=0.25)

    ground_truth_ax = fig.add_subplot(grid[:, 0])
    ground_truth_ax.imshow(read_picture(correct_image))
    ground_truth_ax.axis("off")
    ground_truth_ax.set_title(
        "Ground-truth image\n" + filenames[correct_image],
        fontsize=10,
        color=GREEN,
    )

    record = {
        "direction": "T2I",
        "case": case_name,
        "query_index": int(query_index),
        "query_caption": captions[query_index],
        "ground_truth_filename": filenames[correct_image],
        "models": {},
    }

    rank_labels = []

    for row, model_name in enumerate(labels):
        top5, correct_rank = ranking_for_query(
            "T2I", query_index, model_name
        )
        rank_labels.append(
            f"{labels[model_name]}: GT rank = {correct_rank}"
        )

        retrieved = []
        for column, image_index in enumerate(top5, start=1):
            image_index = int(image_index)
            hit = image_index == correct_image
            similarity = float(
                scores[model_name][image_index, query_index]
            )

            ax = fig.add_subplot(grid[row, column])
            ax.imshow(read_picture(image_index))
            ax.axis("off")
            ax.set_title(
                f"{'CLIP' if row == 0 else 'LoRA'} "
                f"#{column} | {'GT' if hit else 'non-GT'}\n"
                f"cosine={similarity:.3f}",
                color=GREEN if hit else RED,
                fontsize=10,
            )

            retrieved.append({
                "rank": column,
                "filename": filenames[image_index],
                "is_ground_truth": bool(hit),
                "similarity": similarity,
            })

        record["models"][model_name] = {
            "ground_truth_rank": correct_rank,
            "top5": retrieved,
        }

    fig.suptitle(
        f"T2I — {case_name}\n"
        + textwrap.fill("Query: " + captions[query_index], width=125)
        + "\n"
        + " | ".join(rank_labels),
        fontsize=12,
        y=1.04,
    )

    path = VIS_OUT / f"T2I_{case_name}.png"
    fig.savefig(path, dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)

    record["figure"] = path.name
    return record


def draw_i2t(query_index, case_name):
    fig = plt.figure(figsize=(18, 9))
    grid = fig.add_gridspec(
        5, 3, width_ratios=[1.15, 2, 2],
        wspace=0.18, hspace=0.3,
    )

    image_ax = fig.add_subplot(grid[:, 0])
    image_ax.imshow(read_picture(query_index))
    image_ax.axis("off")
    image_ax.set_title(
        "Query image\n" + filenames[query_index], fontsize=11
    )

    correct_captions = [
        captions[i]
        for i in np.flatnonzero(caption_to_image == query_index)
    ]
    record = {
        "direction": "I2T",
        "case": case_name,
        "query_index": int(query_index),
        "query_filename": filenames[query_index],
        "ground_truth_captions": correct_captions,
        "models": {},
    }

    rank_labels = []

    for column, model_name in enumerate(labels, start=1):
        top5, correct_rank = ranking_for_query(
            "I2T", query_index, model_name
        )
        rank_labels.append(
            f"{labels[model_name]}: best GT rank = {correct_rank}"
        )

        retrieved = []
        for row, caption_index in enumerate(top5):
            caption_index = int(caption_index)
            hit = int(caption_to_image[caption_index]) == query_index
            similarity = float(
                scores[model_name][query_index, caption_index]
            )

            ax = fig.add_subplot(grid[row, column])
            ax.axis("off")

            heading = (
                f"#{row + 1} | {'GT' if hit else 'non-GT'} "
                f"| cosine={similarity:.3f}"
            )
            ax.text(
                0, 0.98,
                heading + "\n" + textwrap.fill(
                    captions[caption_index], width=60
                ),
                va="top",
                fontsize=10,
                color=GREEN if hit else RED,
                transform=ax.transAxes,
            )
            if row == 0:
                ax.set_title(labels[model_name], fontsize=12)

            retrieved.append({
                "rank": row + 1,
                "caption_index": caption_index,
                "caption": captions[caption_index],
                "is_ground_truth": bool(hit),
                "similarity": similarity,
            })

        record["models"][model_name] = {
            "best_ground_truth_rank": correct_rank,
            "top5": retrieved,
        }

    fig.suptitle(
        f"I2T — {case_name}\n" + " | ".join(rank_labels),
        fontsize=12,
        y=1.02,
    )

    path = VIS_OUT / f"I2T_{case_name}.png"
    fig.savefig(path, dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)

    record["figure"] = path.name
    return record


# 固定规则与随机种子；不根据图片是否“好看”挑选
rng = np.random.default_rng(2026)
case_records = []

for direction in ("T2I", "I2T"):
    baseline = saved_ranks["A_baseline"][direction]
    lora = saved_ranks["C_seed42"][direction]

    categories = {
        "both_top1_success": (baseline == 1) & (lora == 1),
        "lora_top1_improvement": (baseline > 1) & (lora == 1),
        "lora_top5_failure": lora > 5,
    }

    for case_name, condition in categories.items():
        candidates = np.flatnonzero(condition)
        assert len(candidates) > 0, (
            f"{direction}/{case_name} 无可选样例"
        )

        query_index = int(rng.choice(candidates))
        print(
            f"\n{direction} / {case_name}: "
            f"候选={len(candidates)}，选中query={query_index}",
            flush=True,
        )

        if direction == "T2I":
            record = draw_t2i(query_index, case_name)
        else:
            record = draw_i2t(query_index, case_name)

        record["candidate_count"] = int(len(candidates))
        case_records.append(record)

with open(
    VIS_OUT / "case_selection_and_retrievals.json",
    "w", encoding="utf-8",
) as f:
    json.dump(
        {"selection_seed": 2026, "cases": case_records},
        f, ensure_ascii=False, indent=2,
    )

print("\n第13步检索可视化完成：6张PNG及完整选样记录。")
print("保存目录：", VIS_OUT)
print("绿色=对应标注；红色=非对应标注。")

## 20 Ground-truth captions for failure cases

Original source cell index: 19 (zero-based).

In [ ]:
import json

with open(
    VIS_OUT / "case_selection_and_retrievals.json",
    encoding="utf-8",
) as f:
    selected = json.load(f)

with open(TEST_OUT / "test_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

for case in selected["cases"]:
    if case["case"] != "lora_top5_failure":
        continue

    direction = case["direction"]
    query = case["query_index"]
    image_index = (
        metadata["caption_to_image"][query]
        if direction == "T2I" else query
    )

    print(f"\n===== {direction} 失败案例 =====")
    print("正确图片：", metadata["filenames"][image_index])
    if direction == "T2I":
        print("查询文本：", metadata["captions"][query])

    print("该图片的五条正确标注：")
    correct = [
        caption
        for caption, owner in zip(
            metadata["captions"],
            metadata["caption_to_image"],
        )
        if owner == image_index
    ]
    for number, caption in enumerate(correct, 1):
        print(f"{number}. {caption}")

## 21 Parameter and MAC/FLOP accounting

Original source cell index: 20 (zero-based).

In [ ]:
COMPLEXITY_OUT = ROOT / "results" / "complexity"
if (COMPLEXITY_OUT / "params_and_compute.csv").is_file():
    import pandas as pd
    from IPython.display import display
    print("Reading saved compute counts.")
    display(pd.read_csv(COMPLEXITY_OUT / "params_and_compute.csv"))
else:
    print("[复杂度] 开始统计参数量与主要算子MACs", flush=True)

    import gc
    import json
    from collections import defaultdict
    from pathlib import Path

    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from PIL import Image
    from IPython.display import display

    COMPLEXITY_OUT = ROOT / "results" / "complexity"
    COMPLEXITY_OUT.mkdir(parents=True, exist_ok=True)

    with open(TEST_OUT / "test_metadata.json", encoding="utf-8") as f:
        profile_metadata = json.load(f)

    # 使用真实图片及其对应的真实caption；仅用于计算量统计
    profile_filename = profile_metadata["filenames"][0]
    profile_caption = next(
        caption
        for caption, owner in zip(
            profile_metadata["captions"],
            profile_metadata["caption_to_image"],
        )
        if owner == 0
    )

    with Image.open(IMG_DIR / profile_filename) as image:
        profile_inputs = processor(
            images=image.convert("RGB"),
            text=[profile_caption],
            padding="max_length",
            truncation=True,
            max_length=77,
            return_tensors="pt",
        )

    pixels = profile_inputs["pixel_values"].to(DEVICE)
    tokens = {
        key: profile_inputs[key].to(DEVICE)
        for key in ("input_ids", "attention_mask")
    }

    assert pixels.shape == (1, 3, 224, 224)
    assert tokens["input_ids"].shape == (1, 77)

    print("图片：", profile_filename)
    print("Caption：", profile_caption)
    print("图片输入：", tuple(pixels.shape))
    print("文本输入：", tuple(tokens["input_ids"].shape), flush=True)


    def count_encoder_macs(model):
        counter = defaultdict(int)
        handles = []

        def linear_hook(module, args, output):
            # 每个输出向量需要 in_features × out_features 次MAC
            vectors = output.numel() // module.out_features
            counter["linear"] += (
                vectors * module.in_features * module.out_features
            )

        def conv_hook(module, args, output):
            kernel_h, kernel_w = module.kernel_size
            products_per_output = (
                module.in_channels // module.groups
            ) * kernel_h * kernel_w
            counter["conv2d"] += output.numel() * products_per_output

        def lora_hook(module, args, output):
            if not module.enabled:
                return
            x = args[0]
            vectors = x.numel() // module.base.in_features
            counter["lora"] += vectors * (
                module.base.in_features * module.r
                + module.r * module.base.out_features
            )
            counter["lora_calls"] += 1

        def attention_hook(module, args, kwargs, output):
            hidden = kwargs.get("hidden_states")
            if hidden is None:
                hidden = args[0]

            batch, length, width = hidden.shape

            # QK^T 与 attention_weights @ V
            # 使用完整稠密矩阵口径，包括被mask的位置
            counter["attention_matmul"] += (
                2 * batch * length * length * width
            )
            counter["attention_calls"] += 1

        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                handles.append(
                    module.register_forward_hook(linear_hook)
                )
            if isinstance(module, nn.Conv2d):
                handles.append(
                    module.register_forward_hook(conv_hook)
                )
            if isinstance(module, ManualLoRALinear):
                handles.append(
                    module.register_forward_hook(lora_hook)
                )
            if name.endswith(".self_attn"):
                handles.append(
                    module.register_forward_hook(
                        attention_hook, with_kwargs=True
                    )
                )

        try:
            model.float().eval()

            with torch.no_grad():
                counter.clear()
                pooled = model.vision_model(
                    pixel_values=pixels
                ).pooler_output
                image_feature = F.normalize(
                    model.visual_projection(pooled), dim=-1
                )
                image_counts = dict(counter)

                counter.clear()
                pooled = model.text_model(**tokens).pooler_output
                text_feature = F.normalize(
                    model.text_projection(pooled), dim=-1
                )
                text_counts = dict(counter)

            assert torch.isfinite(image_feature).all()
            assert torch.isfinite(text_feature).all()
            assert image_counts["attention_calls"] == 12
            assert text_counts["attention_calls"] == 12
            assert image_counts.get("lora_calls", 0) == 0

            categories = ("linear", "conv2d", "attention_matmul", "lora")

            image_macs = sum(image_counts.get(k, 0) for k in categories)
            text_macs = sum(text_counts.get(k, 0) for k in categories)

            return image_macs, text_macs, image_counts, text_counts

        finally:
            for handle in handles:
                handle.remove()


    rows = []
    details = {}

    for group, label in (
        ("A", "A: Original CLIP"),
        ("B", "B: Projection-only"),
        ("C", "C: LoRA + projection"),
    ):
        print(f"\n正在统计 {label}", flush=True)

        model = load_saved_retrieval_model(
            group, seed=42
        ).to(DEVICE).float().eval()

        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(
            p.numel() for p in model.parameters() if p.requires_grad
        )

        image_macs, text_macs, image_detail, text_detail = (
            count_encoder_macs(model)
        )
        pair_macs = image_macs + text_macs

        if group == "C":
            assert text_detail["lora_calls"] == 24
        else:
            assert text_detail.get("lora_calls", 0) == 0

        row = {
            "Group": group,
            "Model": label,
            "TotalParams": total_params,
            "TrainableParams": trainable_params,
            "TrainablePercent": 100 * trainable_params / total_params,
            "Image_GMACs": image_macs / 1e9,
            "Text77_GMACs": text_macs / 1e9,
            "Pair_GMACs": pair_macs / 1e9,
            "Image_GFLOPs": 2 * image_macs / 1e9,
            "Text77_GFLOPs": 2 * text_macs / 1e9,
            "Pair_GFLOPs": 2 * pair_macs / 1e9,
        }
        rows.append(row)

        details[group] = {
            "image_macs": image_macs,
            "text_macs": text_macs,
            "image_breakdown": image_detail,
            "text_breakdown": text_detail,
        }

        print(
            f"总参数={total_params:,}，"
            f"原训练策略可训练参数={trainable_params:,}"
        )
        print(
            f"图片={image_macs / 1e9:.6f} GMACs，"
            f"文本={text_macs / 1e9:.6f} GMACs",
            flush=True,
        )

        del model
        gc.collect()
        torch.cuda.empty_cache()


    # 核对计数是否完整、一致
    assert rows[0]["TotalParams"] == 151277313
    assert rows[1]["TrainableParams"] == 262144
    assert rows[2]["TrainableParams"] == 458752

    assert details["A"]["image_macs"] == details["B"]["image_macs"]
    assert details["A"]["image_macs"] == details["C"]["image_macs"]
    assert details["A"]["text_macs"] == details["B"]["text_macs"]

    # 24个Q/V LoRA分支，总共196608个LoRA参数，每个token使用一次
    expected_extra_macs = 77 * 196608
    actual_extra_macs = (
        details["C"]["text_macs"] - details["A"]["text_macs"]
    )
    assert actual_extra_macs == expected_extra_macs

    table = pd.DataFrame(rows)
    table.to_csv(
        COMPLEXITY_OUT / "params_and_compute.csv", index=False
    )

    report = {
        "input": {
            "image": [1, 3, 224, 224],
            "text": [1, 77],
            "real_image_filename": profile_filename,
            "real_caption": profile_caption,
        },
        "convention": {
            "flops_per_mac": 2,
            "attention": "dense full-length QK^T and attention @ V",
            "lora": "unmerged; both low-rank linear operations included",
            "included": [
                "Conv2d", "Linear", "attention matrix products",
                "manual LoRA matrix products",
            ],
            "excluded": [
                "bias and residual additions", "normalization",
                "activation", "softmax", "embedding lookup",
                "LoRA scaling", "gallery scoring and ranking",
            ],
            "scope": "theoretical dominant-operator compute; not hardware measurement",
        },
        "results": rows,
        "breakdown": details,
    }

    with open(
        COMPLEXITY_OUT / "compute_counting_details.json",
        "w", encoding="utf-8",
    ) as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print("\n===== 参数量与理论计算量 =====")
    display(table.round(6))

    print(
        "\nLoRA新增计算量："
        f"{actual_extra_macs:,} MACs / 77-token文本"
    )
    print("计数一致性检查通过。")
    print("第14步A完成，结果目录：", COMPLEXITY_OUT)
    print("尚未测量推理速度；以上数值不是耗时或FPS。")

## 22 Single-GPU latency and throughput

Original source cell index: 21 (zero-based).

In [ ]:
SPEED_OUT = ROOT / "results" / "complexity"
if (SPEED_OUT / "inference_speed.csv").is_file():
    import pandas as pd
    from IPython.display import display
    print("Reading saved timing measurements; the benchmark is not rerun.")
    display(pd.read_csv(SPEED_OUT / "inference_speed.csv"))
else:
    print("[测速] 开始准备真实输入与计时函数", flush=True)

    import gc
    import json
    import time
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn.functional as F
    import transformers
    from pathlib import Path
    from PIL import Image
    from tqdm.auto import tqdm
    from IPython.display import display

    DEVICE = torch.device("cuda:0")
    torch.cuda.set_device(DEVICE)
    assert torch.cuda.is_available()

    TEST_OUT = ROOT / "results" / "final_test"
    SPEED_OUT = ROOT / "results" / "complexity"
    SPEED_OUT.mkdir(parents=True, exist_ok=True)

    with open(TEST_OUT / "test_metadata.json", encoding="utf-8") as f:
        speed_metadata = json.load(f)

    # ------------------------------------------------------------
    # 1. 相同的32张真实图片及每图一条真实caption
    #    读图、预处理、CPU→GPU搬运全部在计时之外
    # ------------------------------------------------------------
    speed_filenames = speed_metadata["filenames"][:32]
    first_caption = {}

    for caption, owner in zip(
        speed_metadata["captions"],
        speed_metadata["caption_to_image"],
    ):
        first_caption.setdefault(owner, caption)

    speed_captions = [first_caption[i] for i in range(32)]

    pictures = []
    for filename in speed_filenames:
        with Image.open(IMG_DIR / filename) as image:
            pictures.append(image.convert("RGB"))

    prepared = processor(
        images=pictures,
        text=speed_captions,
        padding="max_length",
        truncation=True,
        max_length=77,
        return_tensors="pt",
    )

    image_batch = prepared["pixel_values"].to(DEVICE).float()
    text_batch = {
        key: prepared[key].to(DEVICE)
        for key in ("input_ids", "attention_mask")
    }

    image_single = image_batch[:1]
    text_single = {
        key: value[:1] for key, value in text_batch.items()
    }

    assert image_batch.shape == (32, 3, 224, 224)
    assert text_batch["input_ids"].shape == (32, 77)

    # 完整去重图片候选库，1000张
    image_gallery_features = torch.load(
        TEST_OUT / "test_image_features_fp32.pt",
        map_location="cpu",
        weights_only=True,
    ).to(DEVICE).float()

    assert image_gallery_features.shape == (1000, 512)
    image_gallery_T = image_gallery_features.T.contiguous()

    print("GPU:", torch.cuda.get_device_name(DEVICE))
    print("编码输入：相同32张真实图片、32条真实caption")
    print("文本统一填充至77 tokens；精度FP32", flush=True)


    # ------------------------------------------------------------
    # 2. 编码函数：包括投影与L2归一化
    # ------------------------------------------------------------
    def encode_image(model, pixels):
        pooled = model.vision_model(
            pixel_values=pixels
        ).pooler_output
        return F.normalize(model.visual_projection(pooled), dim=-1)


    def encode_text(model, tokens):
        pooled = model.text_model(**tokens).pooler_output
        return F.normalize(model.text_projection(pooled), dim=-1)


    # ------------------------------------------------------------
    # 3. 同步墙钟延迟：包含Python调用、GPU执行及完成同步
    # ------------------------------------------------------------
    @torch.inference_mode()
    def measure_latency(fn, label, warmup=20, repeats=100):
        for _ in range(warmup):
            output = fn()
        torch.cuda.synchronize(DEVICE)

        samples_ms = []

        for _ in tqdm(range(repeats), desc=label):
            torch.cuda.synchronize(DEVICE)
            start = time.perf_counter()

            output = fn()
            torch.cuda.synchronize(DEVICE)

            samples_ms.append(
                (time.perf_counter() - start) * 1000
            )

        values = np.asarray(samples_ms)
        return {
            "mean_ms": float(values.mean()),
            "median_ms": float(np.median(values)),
            "p95_ms": float(np.percentile(values, 95)),
            "samples_ms": samples_ms,
        }


    # ------------------------------------------------------------
    # 4. 吞吐量：每轮连续执行10个batch，共5轮
    #    每轮边界同步，避免每个batch都强制等待
    # ------------------------------------------------------------
    @torch.inference_mode()
    def measure_throughput(
        fn, label, batch_size=32, warmup=10, rounds=5, steps=10
    ):
        for _ in range(warmup):
            output = fn()
        torch.cuda.synchronize(DEVICE)

        durations = []

        for _ in tqdm(range(rounds), desc=label):
            torch.cuda.synchronize(DEVICE)
            start = time.perf_counter()

            for _ in range(steps):
                output = fn()

            torch.cuda.synchronize(DEVICE)
            durations.append(time.perf_counter() - start)

        total_items = batch_size * rounds * steps

        return {
            "items_per_second": float(
                total_items / sum(durations)
            ),
            "mean_batch_ms": float(
                1000 * sum(durations) / (rounds * steps)
            ),
            "round_seconds": durations,
            "batch_size": batch_size,
            "rounds": rounds,
            "steps_per_round": steps,
        }


    # 若之前保留的恢复模型仍在GPU，先移至CPU
    if "restored_c_model" in globals():
        restored_c_model.cpu()

    gc.collect()
    torch.cuda.empty_cache()

    speed_rows = []
    raw_measurements = {}

    # ------------------------------------------------------------
    # 5. A/B/C采用相同输入、相同测量次数
    # ------------------------------------------------------------
    for group, feature_label in (
        ("A", "A_baseline"),
        ("B", "B_seed42"),
        ("C", "C_seed42"),
    ):
        print(f"\n===== 开始 {group} 组测速 =====", flush=True)

        current = load_saved_retrieval_model(
            group, seed=42
        ).to(DEVICE).float().eval()

        # 先检查实际输出
        with torch.inference_mode():
            check_image = encode_image(current, image_single)
            check_text = encode_text(current, text_single)

        assert check_image.shape == (1, 512)
        assert check_text.shape == (1, 512)
        assert torch.isfinite(check_image).all()
        assert torch.isfinite(check_text).all()
        del check_image, check_text

        image_latency = measure_latency(
            lambda: encode_image(current, image_single),
            f"{group} 图片编码 B=1",
        )
        text_latency = measure_latency(
            lambda: encode_text(current, text_single),
            f"{group} 文本编码 B=1",
        )

        image_throughput = measure_throughput(
            lambda: encode_image(current, image_batch),
            f"{group} 图片吞吐 B=32",
        )
        text_throughput = measure_throughput(
            lambda: encode_text(current, text_batch),
            f"{group} 文本吞吐 B=32",
        )

        # 当前模型对应的5000条真实测试caption特征
        text_gallery_features = torch.load(
            TEST_OUT / f"{feature_label}_text_features_fp32.pt",
            map_location="cpu",
            weights_only=True,
        ).to(DEVICE).float()

        assert text_gallery_features.shape == (5000, 512)
        text_gallery_T = text_gallery_features.T.contiguous()

        # 轮换64条已编码查询；候选库始终完整
        t2i_queries = [
            text_gallery_features[i:i + 1] for i in range(64)
        ]
        i2t_queries = [
            image_gallery_features[i:i + 1] for i in range(64)
        ]
        positions = {"T2I": 0, "I2T": 0}

        def cached_t2i():
            index = positions["T2I"] % len(t2i_queries)
            positions["T2I"] += 1
            similarity = t2i_queries[index] @ image_gallery_T
            return torch.topk(
                similarity, k=5, dim=-1, largest=True, sorted=True
            )

        def cached_i2t():
            index = positions["I2T"] % len(i2t_queries)
            positions["I2T"] += 1
            similarity = i2t_queries[index] @ text_gallery_T
            return torch.topk(
                similarity, k=5, dim=-1, largest=True, sorted=True
            )

        t2i_latency = measure_latency(
            cached_t2i,
            f"{group} 缓存T2I Top5 / 1000图片",
            repeats=200,
        )
        i2t_latency = measure_latency(
            cached_i2t,
            f"{group} 缓存I2T Top5 / 5000文本",
            repeats=200,
        )

        row = {
            "Group": group,
            "Image_B1_mean_ms": image_latency["mean_ms"],
            "Image_B1_p95_ms": image_latency["p95_ms"],
            "Text77_B1_mean_ms": text_latency["mean_ms"],
            "Text77_B1_p95_ms": text_latency["p95_ms"],
            "Image_B32_images_s": image_throughput["items_per_second"],
            "Text77_B32_texts_s": text_throughput["items_per_second"],
            "T2I_cached_Top5_mean_ms": t2i_latency["mean_ms"],
            "I2T_cached_Top5_mean_ms": i2t_latency["mean_ms"],
        }
        speed_rows.append(row)

        raw_measurements[group] = {
            "image_b1_latency": image_latency,
            "text77_b1_latency": text_latency,
            "image_b32_throughput": image_throughput,
            "text77_b32_throughput": text_throughput,
            "t2i_cached_top5_latency": t2i_latency,
            "i2t_cached_top5_latency": i2t_latency,
        }

        print(
            f"{group} 图片编码：{row['Image_B1_mean_ms']:.3f} ms/张；"
            f"文本编码：{row['Text77_B1_mean_ms']:.3f} ms/条",
            flush=True,
        )
        print(
            f"{group} Batch32吞吐："
            f"{row['Image_B32_images_s']:.2f} images/s，"
            f"{row['Text77_B32_texts_s']:.2f} texts/s",
            flush=True,
        )

        del current, text_gallery_features, text_gallery_T
        del t2i_queries, i2t_queries
        gc.collect()
        torch.cuda.empty_cache()


    # ------------------------------------------------------------
    # 6. 保存表格、原始计时样本与完整测量口径
    # ------------------------------------------------------------
    speed_table = pd.DataFrame(speed_rows)
    speed_table.to_csv(
        SPEED_OUT / "inference_speed.csv", index=False
    )

    speed_report = {
        "environment": {
            "gpu": torch.cuda.get_device_name(DEVICE),
            "device": str(DEVICE),
            "torch": str(torch.__version__),
            "transformers": transformers.__version__,
            "cuda": torch.version.cuda,
            "precision": "float32",
            "autocast": False,
            "compiled": False,
            "lora": "unmerged",
            "checkpoint_seed": 42,
        },
        "protocol": {
            "encoder_image_shape": [1, 3, 224, 224],
            "encoder_text_shape": [1, 77],
            "latency": "synchronized wall-clock; mean, median and p95",
            "encoder_latency_warmup": 20,
            "encoder_latency_repeats": 100,
            "throughput_batch_size": 32,
            "throughput_warmup": 10,
            "throughput_rounds": 5,
            "throughput_steps_per_round": 10,
            "retrieval_latency_warmup": 20,
            "retrieval_latency_repeats": 200,
            "retrieval_topk": 5,
            "t2i_gallery_images": 1000,
            "i2t_gallery_captions": 5000,
            "retrieval_query_bank_size": 64,
            "encoder_scope": "encoder + projection + L2 normalization",
            "retrieval_scope": "cached query/gallery dot product + sorted Top5",
            "outside_timing": [
                "model loading",
                "file reading",
                "image preprocessing",
                "text tokenization",
                "CPU-GPU input transfer",
                "gallery feature extraction",
                "GPU-CPU result transfer",
                "visualization",
            ],
            "input_image_filenames": speed_filenames,
            "input_captions": speed_captions,
        },
        "summary": speed_rows,
        "raw_measurements": raw_measurements,
    }

    with open(
        SPEED_OUT / "inference_speed_details.json",
        "w", encoding="utf-8",
    ) as f:
        json.dump(speed_report, f, ensure_ascii=False, indent=2)

    print("\n===== 单GPU FP32测速结果 =====")
    display(speed_table.round(4))

    print("\n第14步B完成。")
    print("结果目录：", SPEED_OUT)
    print("图片吞吐量单位为images/s，文本吞吐量单位为texts/s。")
    print("结果覆盖GPU驻留输入下的编码与缓存匹配，不含预处理和传输。")

## 23 Paper figures and summary tables

Original source cell index: 22 (zero-based).

In [ ]:
print("[绘图] 开始读取已保存的日志和实验结果", flush=True)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

TEST_OUT = ROOT / "results" / "final_test"
COMPLEXITY_OUT = ROOT / "results" / "complexity"
FIG_OUT = ROOT / "results" / "paper_figures"
FIG_OUT.mkdir(parents=True, exist_ok=True)

SEEDS = (42, 43, 44)
GROUPS = ("A", "B", "C")
COLORS = {"A": "#777777", "B": "#0072B2", "C": "#D55E00"}
LABELS = {
    "A": "A: Original CLIP",
    "B": "B: Projection-only",
    "C": "C: LoRA + projection",
}

plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.facecolor": "white",
})

figure_notes = {}
generated_files = []


def save_figure(fig, name, note):
    for extension in ("png", "pdf"):
        path = FIG_OUT / f"{name}.{extension}"
        fig.savefig(path, dpi=300, bbox_inches="tight")
        assert path.is_file() and path.stat().st_size > 0
        generated_files.append(path.name)

    figure_notes[name] = note
    display(fig)
    plt.close(fig)
    print(f"已保存：{name}.png / .pdf", flush=True)


# ============================================================
# 1. 读取六份训练日志
# ============================================================
histories = {}
training_source = []

for group in ("B", "C"):
    for seed in SEEDS:
        folder = (
            f"B_projection_seed{seed}"
            if group == "B"
            else f"C_lora_r8_seed{seed}"
        )

        with open(
            ROOT / "results" / folder / "summary.json",
            encoding="utf-8",
        ) as f:
            summary = json.load(f)

        assert summary["seed"] == seed
        history = sorted(summary["history"], key=lambda x: x["epoch"])
        assert [x["epoch"] for x in history] == [1, 2, 3, 4, 5]

        histories[(group, seed)] = history

        for item in history:
            training_source.append({
                "Group": group,
                "Seed": seed,
                "Epoch": item["epoch"],
                "TrainLoss": item["train_loss"],
                "Validation_mR": item["val"]["mR"],
                "TrainSeconds": item["train_seconds"],
                "PeakAllocatedGiB": item["peak_allocated_gib"],
            })

pd.DataFrame(training_source).to_csv(
    FIG_OUT / "training_curves_source.csv", index=False
)


# ============================================================
# 2. 从最终测试原始结果计算均值与样本标准差
# ============================================================
with open(TEST_OUT / "completed.json", encoding="utf-8") as f:
    completed = json.load(f)

runs = completed["runs"]
assert len(runs) == 7

metric_names = [
    f"{direction}_{metric}"
    for direction in ("I2T", "T2I")
    for metric in ("R@1", "R@5", "R@10", "MedianRank")
] + ["mR"]


def read_metric(run, metric):
    if metric == "mR":
        return float(run["metrics"]["mR"])
    direction, name = metric.split("_", 1)
    return float(run["metrics"][direction][name])


statistics = {}
statistics_rows = []

for group in GROUPS:
    selected = [run for run in runs if run["group"] == group]

    if group == "A":
        assert len(selected) == 1
    else:
        assert len(selected) == 3
        assert {run["seed"] for run in selected} == set(SEEDS)

    statistics[group] = {}

    for metric in metric_names:
        values = np.array(
            [read_metric(run, metric) for run in selected],
            dtype=float,
        )
        mean = float(values.mean())
        std = float(values.std(ddof=1)) if len(values) > 1 else 0.0

        statistics[group][metric] = (mean, std)
        statistics_rows.append({
            "Group": group,
            "Metric": metric,
            "Mean": mean,
            "SampleStd": std if group != "A" else None,
            "Runs": len(values),
        })

pd.DataFrame(statistics_rows).to_csv(
    FIG_OUT / "test_summary_source.csv", index=False
)

# 与之前保存的汇总CSV核对
previous = pd.read_csv(TEST_OUT / "test_mean_std.csv")
for row in previous.to_dict("records"):
    mean, std = statistics[row["Group"]][row["Metric"]]
    assert np.isclose(mean, row["Mean"], atol=1e-9)
    assert np.isclose(std, row["SampleStd"], atol=1e-9)

params = pd.read_csv(
    COMPLEXITY_OUT / "params_and_compute.csv"
).set_index("Group")

speed = pd.read_csv(
    COMPLEXITY_OUT / "inference_speed.csv"
).set_index("Group")

assert set(params.index) == set(GROUPS)
assert set(speed.index) == set(GROUPS)


# ============================================================
# 图1：训练loss和验证mR，均值±样本标准差
# ============================================================
epochs = np.arange(1, 6)
fig, axes = plt.subplots(
    1, 2, figsize=(7.2, 2.8), constrained_layout=True
)

for group in ("B", "C"):
    losses = np.array([
        [item["train_loss"] for item in histories[(group, seed)]]
        for seed in SEEDS
    ])
    validation = np.array([
        [item["val"]["mR"] for item in histories[(group, seed)]]
        for seed in SEEDS
    ])

    for ax, values in zip(axes, (losses, validation)):
        mean = values.mean(axis=0)
        std = values.std(axis=0, ddof=1)

        ax.plot(
            epochs, mean,
            marker="o" if group == "B" else "s",
            markersize=4,
            color=COLORS[group],
            label=LABELS[group],
        )
        ax.fill_between(
            epochs, mean - std, mean + std,
            color=COLORS[group], alpha=0.18,
        )

for ax in axes:
    ax.set_xlabel("Epoch")
    ax.set_xticks(epochs)
    ax.grid(alpha=0.2)

axes[0].set_title("(a) Training loss")
axes[0].set_ylabel("Contrastive loss")
axes[0].legend(frameon=False)

axes[1].set_title("(b) Validation retrieval")
axes[1].set_ylabel("Validation mR (%)")

save_figure(
    fig,
    "fig01_training_and_validation",
    "B/C curves show mean ± sample standard deviation across "
    "seeds 42, 43 and 44 at each epoch. No smoothing is applied. "
    "Training loss and validation mR use different y-axis scales.",
)


# ============================================================
# 图2：双向Recall，三组并列柱形图
# ============================================================
fig, axes = plt.subplots(
    1, 2, figsize=(7.2, 3.0), sharey=True,
    constrained_layout=True,
)
x = np.arange(3)
width = 0.24

for ax, direction in zip(axes, ("I2T", "T2I")):
    for offset, group in enumerate(GROUPS):
        values = [
            statistics[group][f"{direction}_R@{k}"][0]
            for k in (1, 5, 10)
        ]
        errors = [
            statistics[group][f"{direction}_R@{k}"][1]
            for k in (1, 5, 10)
        ]

        bars = ax.bar(
            x + (offset - 1) * width,
            values,
            width=width,
            color=COLORS[group],
            label=LABELS[group],
            yerr=errors if group != "A" else None,
            capsize=2,
            error_kw={"elinewidth": 0.9},
        )
        ax.bar_label(
            bars, labels=[f"{v:.1f}" for v in values],
            padding=3, fontsize=6.5,
        )

    ax.set_xticks(x, ["R@1", "R@5", "R@10"])
    ax.set_ylim(0, 108)
    ax.set_title(
        "Image-to-text" if direction == "I2T" else "Text-to-image"
    )
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

axes[0].set_ylabel("Recall (%)")
handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, legend_labels,
    loc="upper center", bbox_to_anchor=(0.5, 1.12),
    ncol=3, frameon=False,
)

save_figure(
    fig,
    "fig02_test_bidirectional_recall",
    "Test galleries contain 1000 images and 5000 captions. "
    "A is one deterministic zero-shot baseline; B/C bars show "
    "means across three seeds, with sample standard deviations. "
    "Error bars are not confidence intervals.",
)


# ============================================================
# 图3：可训练参数量与测试mR
# ============================================================
fig, ax = plt.subplots(
    figsize=(3.6, 2.9), constrained_layout=True
)

for group in GROUPS:
    trainable_k = float(params.loc[group, "TrainableParams"]) / 1000
    mean, std = statistics[group]["mR"]

    ax.errorbar(
        trainable_k, mean,
        yerr=std if group != "A" else None,
        fmt={"A": "o", "B": "s", "C": "^"}[group],
        color=COLORS[group],
        markersize=6,
        capsize=3,
    )
    ax.annotate(
        {"A": "A: CLIP", "B": "B: Projection", "C": "C: LoRA + projection"}[group],
        xy=(trainable_k, mean),
        xytext=(-8, 10) if group == "C" else (8, 8),
        textcoords="offset points",
        ha="right" if group == "C" else "left",
        fontsize=8,
    )

ax.set_xlabel("Trainable parameters (thousands)")
ax.set_ylabel("Test mR (%)")
ax.set_xlim(-35, 540)
ax.set_ylim(83, 90.3)
ax.grid(alpha=0.2)

save_figure(
    fig,
    "fig03_trainable_parameters_vs_test_mR",
    "Trainable parameters follow each training strategy. "
    "A has zero trainable parameters but retains the full pretrained model. "
    "B/C mR values are three-seed means with sample standard deviations. "
    "Points represent different strategies, not a rank sweep.",
)


# ============================================================
# 图4：FP32编码延迟和Batch32吞吐
# ============================================================
fig, axes = plt.subplots(
    1, 2, figsize=(7.2, 3.0), constrained_layout=True
)
x = np.arange(2)
width = 0.24

for offset, group in enumerate(GROUPS):
    latency_values = [
        float(speed.loc[group, "Image_B1_mean_ms"]),
        float(speed.loc[group, "Text77_B1_mean_ms"]),
    ]
    throughput_values = [
        float(speed.loc[group, "Image_B32_images_s"]),
        float(speed.loc[group, "Text77_B32_texts_s"]),
    ]

    for ax, values in zip(
        axes, (latency_values, throughput_values)
    ):
        bars = ax.bar(
            x + (offset - 1) * width,
            values, width=width,
            color=COLORS[group], label=LABELS[group],
        )
        ax.bar_label(
            bars,
            labels=[
                f"{v:.2f}" if ax is axes[0] else f"{v:.1f}"
                for v in values
            ],
            padding=3, fontsize=7,
        )

for ax in axes:
    ax.set_xticks(x, ["Image", "Text (77 tokens)"])
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

axes[0].set_title("(a) Encoding latency, batch=1")
axes[0].set_ylabel("Mean latency (ms)")
axes[0].set_ylim(
    0, speed[["Image_B1_mean_ms", "Text77_B1_mean_ms"]]
    .to_numpy().max() * 1.23
)

axes[1].set_title("(b) Encoding throughput, batch=32")
axes[1].set_ylabel("Throughput (images/s or texts/s)")
axes[1].set_ylim(
    0, speed[["Image_B32_images_s", "Text77_B32_texts_s"]]
    .to_numpy().max() * 1.23
)

handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, legend_labels,
    loc="upper center", bbox_to_anchor=(0.5, 1.12),
    ncol=3, frameon=False,
)

save_figure(
    fig,
    "fig04_encoding_latency_and_throughput",
    "Single Tesla T4, FP32, seed42 checkpoints, unmerged LoRA. "
    "Latency bars show synchronized wall-clock means over 100 calls. "
    "Throughput uses five rounds of ten batch32 calls. "
    "Preprocessing, file I/O and CPU-GPU transfer are outside timing. "
    "These are timing measurements, not three-training-seed averages.",
)


# ============================================================
# 3. 输出论文结果表和效率表
# ============================================================
paper_rows = []

for group in GROUPS:
    row = {"Model": LABELS[group]}

    for metric in metric_names:
        mean, std = statistics[group][metric]
        row[metric] = (
            f"{mean:.2f}"
            if group == "A"
            else f"{mean:.2f} ± {std:.2f}"
        )

    paper_rows.append(row)

paper_results = pd.DataFrame(paper_rows)
paper_results.to_csv(
    FIG_OUT / "paper_results_table.csv",
    index=False, encoding="utf-8-sig",
)

efficiency = (
    params.reset_index()
    .merge(
        speed.reset_index(),
        on="Group",
        validate="one_to_one",
    )
)
efficiency.to_csv(
    FIG_OUT / "paper_efficiency_table.csv",
    index=False, encoding="utf-8-sig",
)

with open(
    FIG_OUT / "figure_notes.json", "w", encoding="utf-8"
) as f:
    json.dump(figure_notes, f, ensure_ascii=False, indent=2)

print("\n===== 论文主结果表 =====")
display(paper_results)

print("\n===== 导出文件 =====")
for filename in generated_files:
    print(filename)

print("\nCSV：")
print("training_curves_source.csv")
print("test_summary_source.csv")
print("paper_results_table.csv")
print("paper_efficiency_table.csv")
print("图注与统计口径：figure_notes.json")

print("\n第15步图表生成完成。")
print("保存目录：", FIG_OUT)
print("共4组图、8个PNG/PDF文件，另附汇总表和绘图数据。")
print("未训练模型；未重新执行测试集评估。")

## 24 Training history reconstructed from saved summaries

Original source cell index: 23 (zero-based).

In [ ]:
print("[归档] 从已保存文件恢复展示训练记录，不执行训练", flush=True)

import json
from pathlib import Path
import pandas as pd
from IPython.display import display, JSON

ROOT = PROJECT_ROOT
ARCHIVE_OUT = ROOT / "results" / "submission_metadata"
ARCHIVE_OUT.mkdir(parents=True, exist_ok=True)

payload = {
    "provenance": (
        "Displayed from previously saved experiment summaries. "
        "These are historical training records, not newly executed training."
    ),
    "experiments": {},
}

with open(
    ROOT / "results/baseline_val.json", encoding="utf-8"
) as f:
    payload["baseline_validation"] = json.load(f)

rows = []

for seed in (42, 43, 44):
    for group, prefix in (
        ("B", "B_projection"),
        ("C", "C_lora_r8"),
    ):
        path = (
            ROOT / "results" / f"{prefix}_seed{seed}" / "summary.json"
        )
        with open(path, encoding="utf-8") as f:
            summary = json.load(f)

        assert summary["seed"] == seed
        history = sorted(
            summary["history"], key=lambda item: item["epoch"]
        )
        assert [item["epoch"] for item in history] == [1, 2, 3, 4, 5]

        payload["experiments"][f"{group}_seed{seed}"] = summary

        for item in history:
            rows.append({
                "Group": group,
                "Seed": seed,
                "Epoch": item["epoch"],
                "TrainLoss": item["train_loss"],
                "Val_mR": item["val"]["mR"],
                "Val_I2T_R1": item["val"]["I2T"]["R@1"],
                "Val_T2I_R1": item["val"]["T2I"]["R@1"],
                "TrainSeconds": item["train_seconds"],
                "PeakAllocatedGiB": item["peak_allocated_gib"],
                "SkippedUpdates": item["skipped_updates"],
            })

        print(
            f"{group}_seed{seed}：5轮记录齐全，"
            f"最佳epoch={summary['best_epoch']}，"
            f"验证mR={summary['val_metrics']['mR']:.4f}%"
        )

table = pd.DataFrame(rows)
assert len(table) == 30

print("\n===== 六次训练的30轮历史记录 =====")
with pd.option_context("display.max_rows", None):
    display(table.round(6))

# 将完整原始数值嵌入Notebook输出，包括配置、各轮指标和来源哈希
print("\n===== 完整历史记录，可展开查看 =====")
display(JSON(payload, expanded=False))

table.to_csv(ARCHIVE_OUT / "training_history.csv", index=False)
with open(
    ARCHIVE_OUT / "saved_experiment_records.json",
    "w", encoding="utf-8",
) as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("\n归档完成：6份实验记录、30轮历史及原始验证基线。")
print("完整数据已嵌入当前单元格输出，并另存为JSON/CSV。")
print("没有重新训练或评估模型。")